# 8.1.2 – Analyse Multilabel da Segunda Fase
Este notebook replica o pipeline da segunda fase usando as anotações salvas no app Streamlit, mas adiciona uma análise **multi-label**, em que uma sentença pode pertencer a mais de uma categoria simultaneamente.


In [13]:
import os
import ast
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import plotly.express as px


## Caminhos de entrada/saída
Aponte para o diretório de anotações da segunda fase que deseja analisar.

In [14]:
sentences_file = Path("data/frases-musicas-final-filled.csv")
assignments_file = Path("data/assignments_second_round.csv")
annotations_root = Path("data/annotations")
annotation_subdir = Path("20251203")  # ajuste se quiser outro snapshot
annotations_dir = (annotations_root / annotation_subdir) if annotation_subdir else annotations_root

output_multilabel_file = Path("data/filtered_second_stage_annotations_multilabel.csv")
output_full_stats_file = Path("data/second_stage_annotations_multilabel_full.csv")
annotations_dir


PosixPath('data/annotations/20251203')

## Leitura dos arquivos base

In [15]:
sentences_df = pd.read_csv(sentences_file, dtype={"frase_id": str})
assignments_df = pd.read_csv(assignments_file, dtype={"frase_id": str, "annotator_id": str})
sentences_df.head(), assignments_df.head()


(   music_id  music_frase_id                                             frase  \
 0         1               1    Carolina é uma menina bem difícil de esquecer.   
 1         1               2                Andar bonito e um brilho no olhar.   
 2         1               3  Tem um jeito adolescente que me faz enlouquecer.   
 3         1               4            E um molejo que eu não vou te enganar.   
 4         1               5          Maravilha feminina, meu docinho de pavê.   
 
   frase_id tem_padrao                                  padrao_encontrado  
 0        1     Female  [(1916877163388338700, 3, 6), (191687716338833...  
 1        2          N                                                NaN  
 2        3          N                                                NaN  
 3        4          N                                                NaN  
 4        5          N                                                NaN  ,
   frase_id annotator_id
 0   594474       user01

## Carrega e consolida as anotações por usuário

In [16]:
annotation_files = sorted([
    annotations_dir / f
    for f in os.listdir(annotations_dir)
    if f.endswith(".csv") and f.startswith("annotation_second_round_user")
])
assert annotation_files, "Nenhum arquivo de anotação encontrado no diretório informado."
annotation_files


[PosixPath('data/annotations/20251203/annotation_second_round_user01.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user02.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user03.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user04.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user05.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user06.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user07.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user08.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user09.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user10.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user11.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user12.csv'),
 PosixPath('data/annotations/20251203/annotation_second_round_user13.csv'),
 PosixPath('

In [17]:
annotations_df = pd.concat([pd.read_csv(f, dtype={"frase_id": str}) for f in annotation_files], ignore_index=True)
annotations_df.head()


,user,frase_id,annotation,sentiment
0,user01,594474,"[""valores""]",positiva
1,user01,3645072,"[""valores""]",negativa
2,user01,2461846,"[""papel_social""]",positiva
3,user01,1846293,"[""identidade""]",positiva
4,user01,2384065,"[""comportamentos""]",negativa


## Junta assignments + anotações (mantendo frases não anotadas)

In [18]:
assignments_with_annotations = assignments_df.merge(
    annotations_df,
    left_on=["annotator_id", "frase_id"],
    right_on=["user", "frase_id"],
    how="left"
)
assignments_with_annotations["user"] = assignments_with_annotations["user"].fillna(assignments_with_annotations["annotator_id"])
assignments_with_annotations["annotation"] = assignments_with_annotations["annotation"].fillna("not_annotated")
assignments_with_annotations.head()


,frase_id,annotator_id,user,annotation,sentiment
0,594474,user01,user01,"[""valores""]",positiva
1,594474,user02,user02,"[""papel_social"", ""valores""]",positiva
2,32675,user02,user02,"[""identidade"", ""aparencia"", ""papel_social"", ""h...",negativa
3,32675,user03,user03,"[""comportamentos""]",negativa
4,3362704,user03,user03,"[""habilidades""]",positiva


## Gera estatísticas por sentença

In [19]:
annotation_types = [
    "identidade", "aparencia", "papel_social",
    "habilidades", "valores", "comportamentos", "outros"
]

awa = assignments_with_annotations.copy()
awa["annotation_list"] = awa["annotation"].apply(lambda x: ast.literal_eval(x) if x != "not_annotated" else [])

annotation_statistics_df = (
    awa.groupby("frase_id")["annotation_list"]
       .apply(list)
       .reset_index()
)

annotation_statistics_df["total_annotations"] = annotation_statistics_df["annotation_list"].apply(
    lambda lists: sum(len(lst) for lst in lists)
)
annotation_statistics_df["annotators"] = annotation_statistics_df["annotation_list"].apply(len)

def count_annotation(list_of_lists, ann_type):
    return sum(lst.count(ann_type) for lst in list_of_lists)

for ann in annotation_types:
    annotation_statistics_df[ann] = annotation_statistics_df["annotation_list"].apply(
        lambda lists, ann=ann: count_annotation(lists, ann)
    )

def get_majority(list_of_lists):
    flattened = [item for lst in list_of_lists for item in lst]
    if not flattened:
        return None
    most_common = Counter(flattened).most_common(1)[0][0]
    return most_common

annotation_statistics_df["majority_vote"] = annotation_statistics_df["annotation_list"].apply(get_majority)

def agreement_rate(row):
    if row["annotators"] == 0 or not row["majority_vote"]:
        return 0.0
    agreeing = sum(
        1 for lst in row["annotation_list"] if row["majority_vote"] in lst
    )
    return agreeing * 100 / row["annotators"]

annotation_statistics_df["agreement_rate"] = annotation_statistics_df.apply(agreement_rate, axis=1)
annotation_statistics_df.head()


,frase_id,annotation_list,total_annotations,annotators,identidade,aparencia,papel_social,habilidades,valores,comportamentos,outros,majority_vote,agreement_rate
0,1,"[[identidade], [identidade]]",2,2,2,0,0,0,0,0,0,identidade,100.0
1,100008,"[[], [habilidades, comportamentos]]",2,2,0,0,0,1,0,1,0,habilidades,50.0
2,100137,"[[identidade, aparencia], [identidade, aparenc...",4,2,2,2,0,0,0,0,0,identidade,100.0
3,1001454,"[[identidade], [identidade]]",2,2,2,0,0,0,0,0,0,identidade,100.0
4,1001532,"[[identidade], [identidade]]",2,2,2,0,0,0,0,0,0,identidade,100.0


## Seleção multi-label baseada em limiar

In [20]:
MIN_SHARE = 0.4   # >= 40% dos anotadores
MIN_VOTES = 2     # e pelo menos 2 votos absolutos

for ann in annotation_types:
    share_col = f"{ann}_share"
    annotation_statistics_df[share_col] = annotation_statistics_df.apply(
        lambda row, ann=ann: (row[ann] / row["annotators"]) if row["annotators"] else 0.0,
        axis=1
    )

def choose_labels(row):
    selected = []
    for ann in annotation_types:
        if row["annotators"] == 0:
            continue
        if row[ann] >= MIN_VOTES and row[f"{ann}_share"] >= MIN_SHARE:
            selected.append(ann)
    if not selected and row["majority_vote"]:
        selected = [row["majority_vote"]]
    return selected

annotation_statistics_df["multi_vote"] = annotation_statistics_df.apply(choose_labels, axis=1)
annotation_statistics_df["n_labels_multi"] = annotation_statistics_df["multi_vote"].apply(len)
annotation_statistics_df[["frase_id", "multi_vote", "n_labels_multi"]].head()


,frase_id,multi_vote,n_labels_multi
0,1,[identidade],1
1,100008,[habilidades],1
2,100137,"[identidade, aparencia]",2
3,1001454,[identidade],1
4,1001532,[identidade],1


## Junta informações da sentença e salva CSVs

In [21]:
annotation_statistics_df = annotation_statistics_df.merge(
    sentences_df[["frase_id", "frase"]],
    on="frase_id",
    how="left"
)

cols_to_export = [
    "frase_id", "frase", "multi_vote", "majority_vote", "agreement_rate",
    "annotators", "total_annotations", "n_labels_multi"
]
export_df = annotation_statistics_df[cols_to_export].copy()
export_df["multi_vote"] = export_df["multi_vote"].apply(lambda lst: json.dumps(lst, ensure_ascii=False))
export_df.to_csv(output_multilabel_file, index=False)
annotation_statistics_df.to_csv(output_full_stats_file, index=False)
output_multilabel_file, output_full_stats_file


(PosixPath('data/filtered_second_stage_annotations_multilabel.csv'),
 PosixPath('data/second_stage_annotations_multilabel_full.csv'))

## Distribuição do número de labels por sentença

In [22]:
labels_per_sentence = (
    annotation_statistics_df["n_labels_multi"].value_counts().sort_index().reset_index()
)
labels_per_sentence.columns = ["n_labels", "sentencas"]
labels_per_sentence["n_labels_label"] = labels_per_sentence["n_labels"].astype(str)

fig = px.bar(
    labels_per_sentence,
    x="n_labels_label",
    y="sentencas",
    text="sentencas",
    title="Quantidade de sentenças por número de rótulos atribuídos",
    color="n_labels_label",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(
    texttemplate="%{text}",
    textposition="outside",
    hovertemplate="# de rótulos: %{x}<br>Sentenças: %{y}<extra></extra>",
)
max_y = labels_per_sentence["sentencas"].max() if not labels_per_sentence.empty else 0
fig.update_layout(
    template="simple_white",
    showlegend=False,
    xaxis_title="# de rótulos",
    yaxis_title="Sentenças",
    title_x=0.5,
    uniformtext_minsize=12,
    uniformtext_mode="hide",
    margin=dict(l=40, r=20, t=60, b=80),
    height=450,
)
fig.update_yaxes(range=[0, max_y * 1.2 if max_y else 1])
fig.show()


## Frequência de cada categoria na análise multi-label

In [23]:
LABELS = {
    "identidade": "Identidade",
    "aparencia": "Aparência",
    "papel_social": "Papel social",
    "habilidades": "Habilidades/Competências",
    "valores": "Valores/Crenças",
    "comportamentos": "Comportamentos",
    "outros": "Outros",
}

multi_label_long = annotation_statistics_df[["frase_id", "multi_vote"]].explode("multi_vote")
multi_label_long = multi_label_long.dropna(subset=["multi_vote"])
label_counts = (
    multi_label_long["multi_vote"].value_counts()
    .reindex(annotation_types, fill_value=0)
    .reset_index()
)
label_counts.columns = ["categoria", "sentencas"]
label_counts = label_counts.sort_values("sentencas", ascending=False).copy()
label_counts["categoria_label"] = label_counts["categoria"].map(LABELS)

fig = px.bar(
    label_counts,
    x="categoria",
    y="sentencas",
    text="sentencas",
    title="Sentenças marcadas por categoria (multi-label)",
    color="categoria",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(
    texttemplate="%{text}",
    textposition="outside",
    hovertemplate="<b>%{customdata[0]}</b><br>Sentenças: %{y}<extra></extra>",
    customdata=label_counts[["categoria_label"]].values,
)
max_y = label_counts["sentencas"].max() if not label_counts.empty else 0
fig.update_layout(
    template="simple_white",
    showlegend=False,
    xaxis_title="Categoria",
    yaxis_title="Sentenças",
    title_x=0.5,
    uniformtext_minsize=12,
    uniformtext_mode="hide",
    margin=dict(l=40, r=20, t=60, b=80),
    height=500,
    xaxis=dict(
        tickangle=-30,
        tickmode="array",
        tickvals=label_counts["categoria"].tolist(),
        ticktext=label_counts["categoria_label"].tolist(),
        categoryorder='array',
        categoryarray=label_counts["categoria"].tolist(),
    ),
)
fig.update_yaxes(range=[0, max_y * 1.25 if max_y else 1])
fig.show()


## Taxa de acordo das sentenças destacadas como multi-label

In [24]:
mean_agreement = annotation_statistics_df["agreement_rate"].mean()
fig = px.histogram(
    annotation_statistics_df,
    x="agreement_rate",
    nbins=20,
    title="Distribuição da taxa de acordo (%)",
    color_discrete_sequence=['#4a90e2'],
)
fig.update_traces(
    hovertemplate="Agreement: %{x:.1f}%<br>Sentenças: %{y}<extra></extra>"
)
fig.update_layout(
    template="simple_white",
    xaxis_title="Agreement (%)",
    yaxis_title="Sentenças",
    bargap=0.08,
    title_x=0.5,
    margin=dict(l=40, r=20, t=60, b=60),
    height=450,
)
if pd.notna(mean_agreement):
    fig.add_vline(
        x=mean_agreement,
        line_color="#ff7f0e",
        line_dash="dash",
        annotation_text=f"Média: {mean_agreement:.1f}%",
        annotation_position="top left",
    )
fig.show()
